In [2]:
import os
import json
import datetime
import random
import zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    get_cosine_schedule_with_warmup,
)
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed = 42
set_seed(seed)

data_dir = r"C:\Users\X\Desktop\X\ALTA\alta2026_public_data"
root_dir = r"C:\Users\X\Desktop\X\ALTA"
test_csv = r"C:\Users\X\Desktop\X\ALTA\sample_test\answer.csv"

model_name = "microsoft/deberta-v3-base"
max_length = 256
grad_accum_steps = 2

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

task_configs = {
    "sentiment": {
        "epochs": 8,
        "batch_size": 16,
        "lr": 1.5e-5,
        "patience": 3,
        "label_smoothing": 0.05,
        "layer_decay": 0.9,
        "classifier_dropout": 0.1,
    },
    "sarcasm": {
        "epochs": 12,
        "batch_size": 16,
        "lr": 1e-5,
        "patience": 4,
        "label_smoothing": 0.1,
        "layer_decay": 0.85,
        "classifier_dropout": 0.2,
    },
}

run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + f"_{model_name.split('/')[-1]}"
run_dir = os.path.join(root_dir, "runs", run_id)
work_dir = os.path.join(run_dir, "tf_runs")
model_out_dir = os.path.join(run_dir, "final_models")
metrics_dir = os.path.join(run_dir, "metrics")
out = os.path.join(run_dir, "answer.csv")
submission_zip = os.path.join(run_dir, "submission.zip")

for d in (run_dir, work_dir, model_out_dir, metrics_dir):
    os.makedirs(d, exist_ok=True)

run_config = {
    "run_id": run_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "model_name": model_name,
    "max_length": max_length,
    "grad_accum_steps": grad_accum_steps,
    "seed": seed,
    "task_configs": task_configs,
    "data_dir": data_dir,
    "use_bf16": use_bf16,
    "use_fp16": use_fp16,
}
with open(os.path.join(run_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)


class TextDataset(Dataset):
    def __init__(self, encodings, labels=None, sample_weights=None):
        self.encodings = encodings
        self.labels = labels
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        if self.sample_weights is not None:
            item["sample_weight"] = torch.tensor(self.sample_weights[idx], dtype=torch.float)
        return item


class WeightedCollator(DataCollatorWithPadding):
    def __call__(self, features):
        weights = [f.pop("sample_weight") for f in features if "sample_weight" in f]
        batch = super().__call__(features)
        if weights:
            batch["sample_weight"] = torch.stack(weights)
        return batch


def make_weighted_trainer(label_smoothing):
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            weight = inputs.pop("sample_weight", None)
            outputs = model(**inputs)
            logits = outputs.logits
            per_sample = torch.nn.functional.cross_entropy(
                logits, labels, reduction="none", label_smoothing=label_smoothing
            )
            if weight is not None:
                weight = weight.to(logits.device)
                loss = (per_sample * weight).sum() / weight.sum()
            else:
                loss = per_sample.mean()
            return (loss, outputs) if return_outputs else loss

    return WeightedTrainer


def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


# def dialect_class_weights(labels, variety):
#     labels = np.array(labels)
#     variety = np.array(variety)
#     weights = np.ones(len(labels), dtype=np.float32)
#
#     for v in np.unique(variety):
#         mask = variety == v
#         cw = compute_class_weight(
#             class_weight="balanced",
#             classes=np.array([0, 1]),
#             y=labels[mask]
#         )
#         for c in (0, 1):
#             weights[mask & (labels == c)] = cw[c]
#
#     return weights

def dialect_class_weights(labels, variety, extra_multiplier=None):
    labels = np.array(labels)
    variety = np.array(variety)
    weights = np.ones(len(labels), dtype=np.float32)

    for v in np.unique(variety):
        mask = variety == v
        cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=labels[mask])
        for c in (0, 1):
            w = cw[c]
            if extra_multiplier and (v, c) in extra_multiplier:
                w *= extra_multiplier[(v, c)]
            weights[mask & (labels == c)] = w

    return weights


def make_compute_metrics(variety_array):
    varieties = sorted(set(variety_array))

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        result = {
            "macro_f1": f1_score(labels, preds, average="macro")
        }

        per_variety = []

        for v in varieties:
            mask = variety_array == v
            f1 = f1_score(
                labels[mask],
                preds[mask],
                average="macro"
            )
            result[f"macro_f1_{v}"] = f1
            per_variety.append(f1)

        result["weak_macro_f1"] = min(per_variety)

        return result

    return compute_metrics


def build_optimizer(model, base_lr, layer_decay, weight_decay=0.01):
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]
    num_layers = model.config.num_hidden_layers
    groups = {}

    def layer_id_of(name):
        if "embeddings" in name:
            return 0
        if ".layer." in name:
            return int(name.split(".layer.")[1].split(".")[0]) + 1
        return num_layers + 1

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        layer_id = layer_id_of(name)
        scale = layer_decay ** (num_layers + 1 - layer_id)
        decay = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        key = (layer_id, decay)
        groups.setdefault(key, {"params": [], "lr": base_lr * scale, "weight_decay": decay})
        groups[key]["params"].append(param)

    return torch.optim.AdamW(list(groups.values()), lr=base_lr)


def train_one_task(task, cfg, train_df, valid_df, test_df, tokenizer, device):
    train_enc = tokenizer(
        list(train_df["text"]),
        truncation=True,
        max_length=max_length
    )

    valid_enc = tokenizer(
        list(valid_df["text"]),
        truncation=True,
        max_length=max_length
    )

    test_enc = tokenizer(
        list(test_df["text"]),
        truncation=True,
        max_length=max_length
    )

    train_labels = train_df[task].tolist()

    #sample_weights = dialect_class_weights(
        #train_labels,
        #train_df["variety"].tolist()
    #)

    sample_weights = dialect_class_weights(
    train_labels, train_df["variety"].tolist(),
    extra_multiplier={("en-AU", 1): 1.6}
    ) 

    train_ds = TextDataset(
        train_enc,
        train_labels,
        sample_weights
    )

    valid_ds = TextDataset(
        valid_enc,
        valid_df[task].tolist()
    )

    test_ds = TextDataset(
        test_enc
    )

    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    config.classifier_dropout = cfg["classifier_dropout"]
    config.hidden_dropout_prob = cfg["classifier_dropout"]

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        torch_dtype=torch.float32,
    )

    model.to(device)

    assert all(p.dtype == torch.float32 for p in model.parameters()), \
        "Model has non-fp32 parameters before training starts"

    collator = WeightedCollator(tokenizer)

    optimizer = build_optimizer(model, cfg["lr"], cfg["layer_decay"])

    steps_per_epoch = max(1, len(train_ds) // (cfg["batch_size"] * grad_accum_steps))
    total_steps = steps_per_epoch * cfg["epochs"]
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    training_args = TrainingArguments(
        output_dir=os.path.join(work_dir, task),
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"] * 2,
        gradient_accumulation_steps=grad_accum_steps,
        learning_rate=cfg["lr"],
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="weak_macro_f1",
        greater_is_better=True,
        logging_steps=50,
        fp16=use_fp16,
        bf16=use_bf16,
        seed=seed,
        report_to=[]
    )

    WeightedTrainer = make_weighted_trainer(cfg["label_smoothing"])

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        compute_metrics=make_compute_metrics(
            valid_df["variety"].to_numpy()
        ),
        optimizers=(optimizer, scheduler),
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=cfg["patience"]
            )
        ],
    )

    trainer.train()

    save_path = os.path.join(model_out_dir, task)
    os.makedirs(save_path, exist_ok=True)
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    with open(os.path.join(metrics_dir, f"{task}_trainer_log_history.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

    valid_preds = trainer.predict(valid_ds)
    test_preds = trainer.predict(test_ds)

    return (
        np.argmax(valid_preds.predictions, axis=-1),
        np.argmax(test_preds.predictions, axis=-1),
    )


def plot_confusion_matrix(cm, class_names, title, save_path):
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)


def compute_all_metrics(gold_df, run_df):
    all_metrics = {}
    official_scores = {}
    class_names = ["0", "1"]

    for task in ["sentiment", "sarcasm"]:
        all_metrics[task] = {}
        for dialect in ["en-AU", "en-UK"]:
            gold = gold_df[gold_df["variety"] == dialect][task]
            pred = run_df[run_df["variety"] == dialect][task]

            macro_f1 = f1_score(gold, pred, average="macro")
            macro_precision = precision_score(gold, pred, average="macro", zero_division=0)
            macro_recall = recall_score(gold, pred, average="macro", zero_division=0)
            acc = accuracy_score(gold, pred)
            report = classification_report(gold, pred, output_dict=True, zero_division=0)
            cm = confusion_matrix(gold, pred, labels=[0, 1])

            all_metrics[task][dialect] = {
                "accuracy": acc,
                "macro_f1": macro_f1,
                "macro_precision": macro_precision,
                "macro_recall": macro_recall,
                "classification_report": report,
                "confusion_matrix": cm.tolist(),
            }

            official_scores[f"{task}-{dialect}"] = macro_f1

            cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                                  columns=[f"pred_{c}" for c in class_names])
            cm_df.to_csv(os.path.join(metrics_dir, f"cm_{task}_{dialect}.csv"))
            plot_confusion_matrix(
                cm, class_names,
                title=f"{task} — {dialect}",
                save_path=os.path.join(metrics_dir, f"cm_{task}_{dialect}.png"),
            )

        gold_all = gold_df[task]
        pred_all = run_df[task]
        cm_all = confusion_matrix(gold_all, pred_all, labels=[0, 1])
        all_metrics[task]["overall"] = {
            "accuracy": accuracy_score(gold_all, pred_all),
            "macro_f1": f1_score(gold_all, pred_all, average="macro"),
            "macro_precision": precision_score(gold_all, pred_all, average="macro", zero_division=0),
            "macro_recall": recall_score(gold_all, pred_all, average="macro", zero_division=0),
            "classification_report": classification_report(gold_all, pred_all, output_dict=True, zero_division=0),
            "confusion_matrix": cm_all.tolist(),
        }
        cm_df_all = pd.DataFrame(cm_all, index=[f"true_{c}" for c in class_names],
                                  columns=[f"pred_{c}" for c in class_names])
        cm_df_all.to_csv(os.path.join(metrics_dir, f"cm_{task}_overall.csv"))
        plot_confusion_matrix(
            cm_all, class_names,
            title=f"{task} — overall",
            save_path=os.path.join(metrics_dir, f"cm_{task}_overall.png"),
        )

    final = (
        min(official_scores["sentiment-en-AU"], official_scores["sentiment-en-UK"])
        + min(official_scores["sarcasm-en-AU"], official_scores["sarcasm-en-UK"])
    ) / 2

    all_metrics["official_weak_macro_f1_per_task_dialect"] = official_scores
    all_metrics["official_final_score"] = final

    return all_metrics, final


device = get_device()
print(f"Run ID: {run_id}")
print(f"Outputs will be written under: {run_dir}")
print(f"Using device: {device}")
print(f"Precision: bf16={use_bf16} fp16={use_fp16}")

train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
valid_df = pd.read_csv(os.path.join(data_dir, "valid.csv"))
test_df = pd.read_csv(test_csv)

tokenizer = AutoTokenizer.from_pretrained(model_name)

valid_preds = {}
test_preds = {}

for task in ["sentiment", "sarcasm"]:
    print(f"\nFine-tuning for task: {task}")
    valid_preds[task], test_preds[task] = train_one_task(
        task, task_configs[task], train_df, valid_df, test_df, tokenizer, device
    )

valid_answer_df = valid_df[["source", "variety", "text"]].copy()
valid_answer_df["sentiment"] = valid_preds["sentiment"]
valid_answer_df["sarcasm"] = valid_preds["sarcasm"]

test_answer_df = test_df[["source", "variety", "text"]].copy()
test_answer_df["sentiment"] = test_preds["sentiment"]
test_answer_df["sarcasm"] = test_preds["sarcasm"]
test_answer_df.to_csv(out, index=False)
print(f"\nWrote test predictions to {out}")

with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(out, arcname="answer.csv")
print(f"Wrote submission zip to {submission_zip}")

all_metrics, final = compute_all_metrics(valid_df, valid_answer_df)

with open(os.path.join(metrics_dir, "all_metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

print("\nOfficial style scores (weak macro F1) on the public valid set:")
for k, v in all_metrics["official_weak_macro_f1_per_task_dialect"].items():
    print(f"  {k}: {v:.4f}")
print(f"  final score: {final:.4f}")

print(f"\nAll metrics, confusion matrices, and models saved under:\n  {run_dir}")

Run ID: 20260923-013123_deberta-v3-base
Outputs will be written under: C:\Users\X\Desktop\X\ALTA\runs\20260923-013123_deberta-v3-base
Using device: cuda
Precision: bf16=True fp16=False

Fine-tuning for task: sentiment


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

{'loss': '1.392', 'grad_norm': '2.623', 'learning_rate': '1.364e-06', 'epoch': '0.289'}
{'loss': '1.213', 'grad_norm': '6.262', 'learning_rate': '2.755e-06', 'epoch': '0.578'}
{'loss': '0.5847', 'grad_norm': '17.61', 'learning_rate': '3.812e-06', 'epoch': '0.8671'}
{'eval_loss': '0.2485', 'eval_macro_f1': '0.9259', 'eval_macro_f1_en-AU': '0.9016', 'eval_macro_f1_en-UK': '0.9481', 'eval_weak_macro_f1': '0.9016', 'eval_runtime': '5.491', 'eval_samples_per_second': '137.9', 'eval_steps_per_second': '4.371', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.568', 'grad_norm': '30.48', 'learning_rate': '3.789e-06', 'epoch': '1.156'}
{'loss': '0.4553', 'grad_norm': '12.59', 'learning_rate': '3.736e-06', 'epoch': '1.445'}
{'loss': '0.4748', 'grad_norm': '5.015', 'learning_rate': '3.654e-06', 'epoch': '1.734'}
{'eval_loss': '0.2456', 'eval_macro_f1': '0.9366', 'eval_macro_f1_en-AU': '0.9081', 'eval_macro_f1_en-UK': '0.9636', 'eval_weak_macro_f1': '0.9081', 'eval_runtime': '5.084', 'eval_samples_per_second': '148.9', 'eval_steps_per_second': '4.721', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (25 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1120', 'train_samples_per_second': '59.24', 'train_steps_per_second': '1.853', 'train_loss': '0.8292', 'epoch': '7'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Wrote test predictions to C:\Users\X\Desktop\X\ALTA\runs\20260923-013123_deberta-v3-base\answer.csv
Wrote submission zip to C:\Users\X\Desktop\X\ALTA\runs\20260923-013123_deberta-v3-base\submission.zip

Official style scores (weak macro F1) on the public valid set:
  sentiment-en-AU: 0.9107
  sentiment-en-UK: 0.9507
  sarcasm-en-AU: 0.6466
  sarcasm-en-UK: 0.7148
  final score: 0.7787

All metrics, confusion matrices, and models saved under:
  C:\Users\X\Desktop\X\ALTA\runs\20260923-013123_deberta-v3-base
